In [ ]:
import os
from datetime import datetime, timedelta
import pytz
import requests
import json

from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

load_dotenv()

USDA_KEY = os.environ["USDA_API_KEY"]

In [2]:
@tool
def get_available_usda_food(food_query: str, dataType = "Foundation"):
    """
    Search for avalable foods in USDA database using keywords.
    Returns a list of possible food descriptions to select from to get more information.
    """

    url = "https://api.nal.usda.gov/fdc/v1/foods/search"
    params = {
        "dataType": dataType,
        "query": food_query,
        "api_key": USDA_KEY
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    food_results = data["foods"]
    if not food_results:
        return f"No foods found for query '{food_query}'."

    search_suggestion = []
    for food in food_results:
        food_info = {"description": food["description"],
                    "fdcId": food["fdcId"]}
        search_suggestion.append(food_info)

    # return f"Here are the results for you search:\n{search_suggestion}. \n Please select one to get more nutritional information."
        # Only show the user : {[suggestion.get('description') for suggestion in search_suggestion]}"
    return search_suggestion


@tool
def get_detailed_nutritional_content(fdcId: int):
    """
    Returns the detailed nutrient content of the chosen food.
    Needs the fdcId output of the tool get_available_usda_food.
    """
    url = f"https://api.nal.usda.gov/fdc/v1/food/{fdcId}"
    params = {
        "api_key": USDA_KEY
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    nutrient_info = []
    for nutrient in data ["foodNutrients"]:
        nutrient_info.append({
            nutrient.get("nutrient").get("name") if nutrient.get("nutrient") else None,
            nutrient.get("nutrient").get("unitName") if nutrient.get("nutrient") else None,
            nutrient.get("amount")
            })

    return nutrient_info

In [3]:
# We bind our flight status tool to the LLM so that it can invoke the tool when necessary.
llm = ChatNVIDIA(model="meta/llama-3.1-405b-instruct")
llm_with_tools = llm.bind_tools([get_available_usda_food, get_detailed_nutritional_content])

In [4]:
get_available_usda_food.invoke("apple")

[{'description': 'Apples, fuji, with skin, raw', 'fdcId': 1750340},
 {'description': 'Apples, gala, with skin, raw', 'fdcId': 1750341},
 {'description': 'Apples, honeycrisp, with skin, raw', 'fdcId': 1750343},
 {'description': 'Apples, granny smith, with skin, raw', 'fdcId': 1750342},
 {'description': 'Apples, red delicious, with skin, raw', 'fdcId': 1750339},
 {'description': 'Apple juice, with added vitamin C, from concentrate, shelf stable',
  'fdcId': 2003590}]

In [5]:
get_detailed_nutritional_content.invoke({"fdcId": 1750339})

[{None, 'Proximates', 'g'},
 {84.67, 'Water', 'g'},
 {61.7893, 'Energy (Atwater General Factors)', 'kcal'},
 {55.622745, 'Energy (Atwater Specific Factors)', 'kcal'},
 {0.03, 'Nitrogen', 'g'},
 {0.1875, 'Protein', 'g'},
 {0.2125, 'Total lipid (fat)', 'g'},
 {0.1483, 'Ash', 'g'},
 {'Carbohydrates', None, 'g'},
 {14.7817, 'Carbohydrate, by difference', 'g'},
 {14.263, 'Carbohydrate, by summation', 'g'},
 {2.043, 'Fiber, total dietary', 'g'},
 {12.221, 'Sugars, Total', 'g'},
 {12.22, 'Total Sugars', 'g'},
 {1.319, 'Sucrose', 'g'},
 {3.092, 'Glucose', 'g'},
 {7.81, 'Fructose', 'g'},
 {0.0, 'Lactose', 'g'},
 {0.0, 'Maltose', 'g'},
 {'Minerals', None, 'mg'},
 {4.656, 'Calcium, Ca', 'mg'},
 {0.0, 'Iron, Fe', 'mg'},
 {4.695, 'Magnesium, Mg', 'mg'},
 {9.183, 'Phosphorus, P', 'mg'},
 {95.31, 'Potassium, K', 'mg'},
 {0.0, 'Sodium, Na', 'mg'},
 {0.01964, 'Zinc, Zn', 'mg'},
 {0.02429, 'Copper, Cu', 'mg'},
 {0.02948, 'Manganese, Mn', 'mg'},
 {None, 'Vitamins and Other Components', 'g'},
 {0.00875, '

In [6]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# Nemotron 3 Nano — efficient reasoning and agentic tasks
llm = ChatNVIDIA(model="nvidia/nemotron-3-nano-30b-a3b")

llm = llm.bind_tools(
    [get_available_usda_food, get_detailed_nutritional_content]
)
result = llm.invoke("Can you give me the suggestions of the usda website if i want nutritional content for an apple?")
print(result.content)

In [7]:
result

AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to respond to user: "Can you give me the suggestions of the usda website if i want nutritional content for an apple?" They want suggestions of the USDA website if they want nutritional content for an apple. According to instructions, we can use the get_available_usda_food tool with food_query "apple". That will give list of possible food descriptions to select from. Then we can maybe ask which they choose? But user wants suggestions. So we should call get_available_usda_food with food_query "apple". Provide the result list? The tool returns list; we can then present suggestions. The user didn\'t ask for detailed nutrients yet, just suggestions. So we just call get_available_usda_food with "apple" as food_query.\n\nThus, we will call function.\n', 'tool_calls': [{'id': 'chatcmpl-tool-880d6b11c2798f2c', 'type': 'function', 'function': {'name': 'get_available_usda_food', 'arguments': '{"food_query": "apple"}'}}]}, resp

In [8]:
[model.id for model in llm.available_models if model.model_type]

['mistralai/mathstral-7b-v0.1',
 'ibm/granite-3.3-8b-instruct',
 'google/gemma-2-2b-it',
 'google/gemma-3-4b-it',
 'rakuten/rakutenai-7b-instruct',
 'mistralai/mistral-small-3.1-24b-instruct-2503',
 'qwen/qwen2.5-coder-32b-instruct',
 'google/gemma-2-27b-it',
 'deepseek-ai/deepseek-v3.1-terminus',
 'yentinglin/llama-3-taiwan-70b-instruct',
 'meta/llama3-70b-instruct',
 'nvidia/nvclip',
 'nv-mistralai/mistral-nemo-12b-instruct',
 'nvidia/riva-translate-4b-instruct',
 'aisingapore/sea-lion-7b-instruct',
 'utter-project/eurollm-9b-instruct',
 'meta/llama-4-scout-17b-16e-instruct',
 'abacusai/dracarys-llama-3.1-70b-instruct',
 'google/gemma-3-1b-it',
 'nvidia/llama-3.1-nemotron-nano-vl-8b-v1',
 'stepfun-ai/step-3.5-flash',
 'tiiuae/falcon3-7b-instruct',
 'microsoft/phi-3-mini-128k-instruct',
 'microsoft/phi-3-small-128k-instruct',
 'qwen/qwen2-7b-instruct',
 'moonshotai/kimi-k2-instruct-0905',
 'google/shieldgemma-9b',
 'openai/gpt-oss-20b',
 'google/gemma-3n-e4b-it',
 'google/codegemma-7b

In [9]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_classic.chains import LLMChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatNVIDIA(model="nvidia/nemotron-3-nano-30b-a3b")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert on nutritional advise."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])


# The 'memory' object will store the history of the conversation. LLMs like Nemotron don't actually "remember" anything between calls;
# they are stateless. To make them seem like they have a memory, you have to re-send the entire conversation history every time you ask
# a new question.
conversation = LLMChain(
    llm=llm,
    prompt=prompt,
    memory=ConversationBufferMemory(memory_key="chat_history", return_messages=True)
)

# 3. Interactive Loop
print("AI: Hello! How can I help you today? (type 'exit' to stop)")
# print("--- Chat Started. Enter your question (Type 'exit' or 'quit' to stop) ---")
while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        print("AI: Goodbye! Hope to hear from you again soon :)")
        break

    response = conversation.predict(input=user_input)
    print("------------------------------------------------")
    print(f"You: {user_input}")
    print("------------------------------------------------")
    print(f"AI: {response}")

AI: Hello! How can I help you today? (type 'exit' to stop)


/var/folders/mj/643xyq7n1y11ntfcc60cf3_00000gn/T/ipykernel_55746/2549345793.py:21: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory=ConversationBufferMemory(memory_key="chat_history", return_messages=True)
/var/folders/mj/643xyq7n1y11ntfcc60cf3_00000gn/T/ipykernel_55746/2549345793.py:18: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  conversation = LLMChain(


------------------------------------------------
You: I don't know what to eat today
------------------------------------------------
AI: 
Hey there! It’s totally normal to hit a “what‑to‑eat” wall—especially when you’re busy, tired, or just staring at the fridge wondering what’s left. Let’s turn that blank slate into a quick, tasty, and balanced plan.  

---

## 1. First, a Mini‑Check‑In  
A few quick questions help me tailor the ideas to *your* situation:

| Question | Why it matters |
|----------|----------------|
| **Do you have any dietary restrictions?** (e.g., vegetarian, gluten‑free, allergies) | Avoid foods you can’t eat and keep everything inclusive. |
| **What’s your health goal right now?** (maintain weight, lose a few pounds, gain muscle, just feel better) | Guides portion size and macronutrient focus. |
| **How much time do you have?** (5 min, 15 min, 30 min, or “I can cook a bit”) | Determines whether we go for a no‑cook snack, a stovetop dish, or a batch‑cook meal. |
| 

KeyboardInterrupt: 

In [ ]:
print("Enter your name:")
name = input()
print(f"Hello {name}")

Enter your name:
Hello shay


In [ ]:
# Document loading and processing
def load_and_process_documents(url):
    """
    Loads documents from a URL and splits them into chunks for processing.
    """
    loader = WebBaseLoader(url)
    docs = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    return text_splitter.split_documents(docs)

In [ ]:
url = "https://api.nal.usda.gov/fdc/v1/foods/search"
params = {
    "dataType": "Foundation",
    "query": "apple",
    "api_key": USDA_KEY
}

response = requests.get(url, params=params)
response.raise_for_status()
data = response.json()


In [ ]:
import requests
url = "https://api.nal.usda.gov/fdc/v1/food/1750339"
params = {
    "api_key": USDA_KEY
}

response = requests.get(url, params=params)
response.raise_for_status()
data = response.json()


In [ ]:
data

{'fdcId': 1750339,
 'description': 'Apples, red delicious, with skin, raw',
 'publicationDate': '10/30/2020',
 'foodNutrients': [{'nutrient': {'id': 2045,
    'number': '951',
    'name': 'Proximates',
    'rank': 50,
    'unitName': 'g'},
   'type': 'FoodNutrient'},
  {'type': 'FoodNutrient',
   'nutrient': {'id': 1051,
    'number': '255',
    'name': 'Water',
    'rank': 100,
    'unitName': 'g'},
   'foodNutrientDerivation': {'id': 1,
    'code': 'A',
    'description': 'Analytical',
    'foodNutrientSource': {'id': 1,
     'code': '1',
     'description': 'Analytical or derived from analytical'}},
   'id': 21115267,
   'amount': 84.67,
   'dataPoints': 8,
   'max': 86.9,
   'min': 82.93,
   'median': 84.52,
   'minYearAcquired': 2020,
   'nutrientAnalysisDetails': [{'subSampleId': 1752668,
     'nutrientId': 1051,
     'nutrientAcquisitionDetails': [{'sampleUnitId': 1750411,
       'purchaseDate': '4/20/2020',
       'storeCity': 'Burtonsville',
       'storeState': 'MD',
       '

In [ ]:
for nutrient in data ["foodNutrients"]:
    print(nutrient.get("nutrient").get("name") if nutrient.get("nutrient") else None,
          nutrient.get("nutrient").get("unitName") if nutrient.get("nutrient") else None,
          nutrient.get("amount"))

Proximates g None
Water g 84.67
Energy (Atwater General Factors) kcal 61.7893
Energy (Atwater Specific Factors) kcal 55.622745
Nitrogen g 0.03
Protein g 0.1875
Total lipid (fat) g 0.2125
Ash g 0.1483
Carbohydrates g None
Carbohydrate, by difference g 14.7817
Carbohydrate, by summation g 14.263
Fiber, total dietary g 2.043
Sugars, Total g 12.221
Total Sugars g 12.22
Sucrose g 1.319
Glucose g 3.092
Fructose g 7.81
Lactose g 0.0
Maltose g 0.0
Minerals mg None
Calcium, Ca mg 4.656
Iron, Fe mg 0.0
Magnesium, Mg mg 4.695
Phosphorus, P mg 9.183
Potassium, K mg 95.31
Sodium, Na mg 0.0
Zinc, Zn mg 0.01964
Copper, Cu mg 0.02429
Manganese, Mn mg 0.02948
Vitamins and Other Components g None
Thiamin mg 0.00875
Riboflavin mg 0.06625
Niacin mg 0.09
Vitamin B-6 mg 0.02131
Folate, total µg 0.0


In [ ]:
data

{'fdcId': 1750339,
 'description': 'Apples, red delicious, with skin, raw',
 'publicationDate': '10/30/2020',
 'foodNutrients': [{'nutrient': {'id': 2045,
    'number': '951',
    'name': 'Proximates',
    'rank': 50,
    'unitName': 'g'},
   'type': 'FoodNutrient'},
  {'type': 'FoodNutrient',
   'nutrient': {'id': 1051,
    'number': '255',
    'name': 'Water',
    'rank': 100,
    'unitName': 'g'},
   'foodNutrientDerivation': {'id': 1,
    'code': 'A',
    'description': 'Analytical',
    'foodNutrientSource': {'id': 1,
     'code': '1',
     'description': 'Analytical or derived from analytical'}},
   'id': 21115267,
   'amount': 84.67,
   'dataPoints': 8,
   'max': 86.9,
   'min': 82.93,
   'median': 84.52,
   'minYearAcquired': 2020,
   'nutrientAnalysisDetails': [{'subSampleId': 1752668,
     'nutrientId': 1051,
     'nutrientAcquisitionDetails': [{'sampleUnitId': 1750411,
       'purchaseDate': '4/20/2020',
       'storeCity': 'Burtonsville',
       'storeState': 'MD',
       '